# L2 detection and ingestion quality metrics (per grid cell)

Run all cells in Google Colab. Input files are harmonized `processed/<sensor>_<year>.parquet` files on Drive. Outputs are the nine `l2_*` tables (one Parquet folder each, read with DuckDB) and the `_runs` row of this batch.

Every metric is grouped by grid cell (`cell`, plus `ilat`, `ilon`, `lat`, `lon`, `lat_band`), not by country: `GRID_DEG = 0.01` (about 1 km, the same value as notebook 02). Rows with an invalid date or invalid coordinates are left out of the cell metrics; QA-01 (global, by instrument) still counts the invalid coordinates. ADD-03 now sees duplicates across country files, because the same detection can appear in the files of two countries. ADD-04 (zero-detection countries) was removed: it is a per-country audit with no per-cell equivalent.

Per-file results are cached in `metrics/_l2_parts/` and the run is resumable. The final tables are consolidated out of memory with DuckDB, ordered by (`ilat`, `ilon`) and partitioned by `lat_band`. `_runs` is written last: a run that stops halfway is not registered.

Use `GITHUB_TOKEN` in Colab Secrets if the repository is private. Set `YEARS` explicitly to require a complete year selection.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')
REPO = Path('/content/repo')
env = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env.update(GIT_CONFIG_COUNT='1',
               GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
               GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], env=env, check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git', str(REPO)], env=env, check=True)
del env, token
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))


In [ ]:
import json
import importlib
import re
import shutil
from datetime import datetime, timezone
import duckdb
import pandas as pd
from metrics import detections, ingestion_qa
from metrics.store import ParquetSink, default_partitions, prepare_table, write_run

importlib.reload(detections)
importlib.reload(ingestion_qa)

SENSORS = ['modis', 'viirs-jpss1']
YEARS = None  # Example: [2018, 2019]. None selects available files.
FORCE = False
CONFIDENCE = ('n', 'h')
TYPE_VALUE = 0
GRID_DEG = 0.01  # cell side in degrees (0.01 is about 1 km); keep equal to notebook 02
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
METRICS.mkdir(parents=True, exist_ok=True)
SINK = ParquetSink(METRICS)
PARTS = METRICS / '_l2_parts'
COLUMNS = ['instrument', 'satellite', 'version', 'latitude', 'longitude',
           'acq_date', 'acq_time', 'acq_datetime_utc', 'daynight', 'type',
           'confidence_raw', 'confidence_class', 'pixel_area_km2', 'frp']
FUNCTIONS = {
    'l2_obs04_night_pct': detections.obs04_night_pct,
    'l2_obs05_frp_median': detections.obs05_frp_median,
    'l2_obs06_frp_p90': detections.obs06_frp_p90,
    'l2_obs07_frp_peak': detections.obs07_frp_peak,
    'l2_obs08_frp_availability': detections.obs08_frp_availability,
    'l2_obs09_type_distribution': detections.obs09_type_distribution,
    'l2_obs10_pixel_area_median': detections.obs10_pixel_area_median,
    'l2_qa01_schema_validation': ingestion_qa.qa01_schema_validation,
    'l2_add03_duplicates': ingestion_qa.add03_duplicates,
}
FILTERED = set(list(FUNCTIONS)[:5])
GLOBAL_TABLES = {'l2_qa01_schema_validation'}  # by instrument, no cell
PARAMETERS = {'confidence': list(CONFIDENCE), 'type_value': TYPE_VALUE, 'grid_deg': GRID_DEG}
CODE_REVISION = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
KEY_COLUMNS = ['cell', 'year', 'instrument', 'satellite', 'version', 'type_label']


def sort_keys(frame):
    return [column for column in KEY_COLUMNS if column in frame.columns]


def atomic_part(frame, path):
    """Per-file cache: categoricals as text, cell columns added, sorted; written atomically."""
    frame = frame.copy()
    for column in frame.columns:
        if isinstance(frame[column].dtype, pd.CategoricalDtype):
            frame[column] = frame[column].astype(object)
    frame = prepare_table(frame, GRID_DEG, sort_by=sort_keys(frame))
    temporary = path.with_suffix('.parquet.tmp')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)


inputs = []
for path in sorted(PROCESSED.glob('*.parquet')):
    match = re.fullmatch(r'(.+)_(\d{4})', path.stem)
    if match and match[1] in SENSORS:
        year = int(match[2])
        if YEARS is None or year in YEARS:
            inputs.append((match[1], year, path))
if not inputs:
    raise FileNotFoundError(f'No selected processed Parquet files in {PROCESSED}')
if YEARS is not None:
    available = {(sensor, year) for sensor, year, _ in inputs}
    missing = [(sensor, year) for sensor in SENSORS for year in YEARS if (sensor, year) not in available]
    if missing:
        raise FileNotFoundError(f'Missing requested sensor/year files: {missing}')

In [ ]:
part_directories = []
for sensor, year, path in inputs:
    directory = PARTS / f'{sensor}_{year}'
    directory.mkdir(parents=True, exist_ok=True)
    marker = directory / '_SUCCESS.json'
    signature = {'source_size': path.stat().st_size, 'source_mtime_ns': path.stat().st_mtime_ns,
                 'parameters': PARAMETERS, 'revision': CODE_REVISION}
    expected = [directory / f'{name}.parquet' for name in FUNCTIONS]
    complete = marker.exists() and all(part.exists() for part in expected)
    cached_signature = None
    if complete:
        try:
            cached_signature = json.loads(marker.read_text())
        except (OSError, ValueError):
            pass  # A damaged completion marker requires recalculation.
    if not FORCE and complete and cached_signature == signature:
        part_directories.append(directory)
        print(f'Reusing completed {sensor} {year}')
        continue
    marker.unlink(missing_ok=True)
    frame = pd.read_parquet(path, columns=COLUMNS)
    for column in ['instrument', 'satellite', 'version', 'daynight', 'confidence_class']:
        frame[column] = frame[column].astype('category')
    for name, function in FUNCTIONS.items():
        if name in FILTERED:
            options = dict(confidence=CONFIDENCE, type_value=TYPE_VALUE, grid_deg=GRID_DEG)
        elif name in GLOBAL_TABLES:
            options = {}
        else:
            options = dict(grid_deg=GRID_DEG)
        result = function(frame, **options)
        if name in GLOBAL_TABLES:
            result.insert(0, 'year', year)
        atomic_part(result, directory / f'{name}.parquet')
    temporary_marker = marker.with_suffix('.json.tmp')
    temporary_marker.write_text(json.dumps(signature, sort_keys=True))
    temporary_marker.replace(marker)
    part_directories.append(directory)
    del frame
    print(f'Processed {sensor} {year}')

In [ ]:
def duckdb_list(files):
    return '[' + ', '.join("'" + str(f).replace('\\', '/').replace("'", "''") + "'" for f in files) + ']'


def consolidate(name, files, check_keys=True):
    """Merge the per-file parts with DuckDB (out of memory), check repeated keys and publish
    the table as a Parquet folder ordered by (ilat, ilon) and partitioned by lat_band."""
    listing = duckdb_list(files)
    source = f'read_parquet({listing}, union_by_name=true)'
    schema = pd.read_parquet(files[0]).iloc[:0]
    keys = sort_keys(schema)
    order = (['ilat', 'ilon'] + [key for key in keys if key != 'cell']) if 'cell' in schema.columns else keys
    partitions = default_partitions(schema)
    con = duckdb.connect()
    try:
        n_rows = con.execute(f'SELECT count(*) FROM {source}').fetchone()[0]
        if check_keys:
            repeated = con.execute(
                f'SELECT count(*) FROM (SELECT 1 FROM {source} GROUP BY {", ".join(keys)} HAVING count(*) > 1)'
            ).fetchone()[0]
            if repeated:
                raise ValueError(f'{name}: {repeated} repeated keys across files')
        if n_rows == 0:
            SINK.write(name, schema, ())
            return 0
        tmp = METRICS / f'{name}.tmp'
        if tmp.exists():
            shutil.rmtree(tmp)
        if partitions:
            part, target = f', PARTITION_BY ({", ".join(partitions)})', tmp
        else:  # without partitions DuckDB writes a single file: put it inside the table folder
            tmp.mkdir(parents=True)
            part, target = '', tmp / 'part-0.parquet'
        destination = target.as_posix().replace("'", "''")
        con.execute(f"COPY (SELECT * FROM {source} ORDER BY {', '.join(order)}) TO '{destination}' "
                    f"(FORMAT PARQUET, COMPRESSION zstd{part}, OVERWRITE_OR_IGNORE)")
    finally:
        con.close()
    SINK._publish(tmp, name)
    return int(n_rows)


tables = {}
for name in FUNCTIONS:
    files = [directory / f'{name}.parquet' for directory in part_directories]
    # QA-01 is global: an unknown instrument can repeat across files, so its keys are not checked.
    tables[name] = consolidate(name, files, check_keys=name not in GLOBAL_TABLES)
    print(f'{name}: {tables[name]} rows')

# _runs is written last: a run that stops halfway is not registered.
write_run(SINK, 'lote2', commit=CODE_REVISION, grid_deg=GRID_DEG, tables=tables,
          params=dict(PARAMETERS, sensor_years=[[sensor, year] for sensor, year, _ in inputs],
                      run_utc=datetime.now(timezone.utc).isoformat(timespec='seconds')))

In [ ]:
for directory in part_directories:
    qa = pd.read_parquet(directory / 'l2_qa01_schema_validation.parquet')
    assert (qa.n_valid + qa.n_invalid).equals(qa.n_rows), directory
    availability = pd.read_parquet(directory / 'l2_obs08_frp_availability.parquet')
    assert availability[['n_frp_usable', 'n_frp_null', 'n_frp_zero', 'n_frp_negative']].sum(axis=1).equals(availability.n_eligible), directory
    types = pd.read_parquet(directory / 'l2_obs09_type_distribution.parquet')
    totals = types.groupby(['cell', 'year', 'instrument', 'satellite', 'version'], observed=True, dropna=False).pct.sum()
    assert totals.sub(100).abs().lt(1e-9).all(), directory
for name, n_rows in tables.items():
    sample = pd.read_parquet(part_directories[0] / f'{name}.parquet')
    print(name, f'{n_rows} rows;', 'columns:', ', '.join(sample.columns))
print(f'L2 completed: {len(tables)} tables in {METRICS}')